In [1]:
# ===============================
# 1) IMPORT LIBRARIES
# ===============================
import os
import PyPDF2
import cohere
import faiss
from sentence_transformers import SentenceTransformer
import numpy as np
from rapidfuzz import process, fuzz
import gradio as gr

print("Step 1: Libraries imported successfully.")


c:\Users\ASUS.SXT412\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Step 1: Libraries imported successfully.


In [ ]:
# ===============================
# 2) CONFIGURATION
# ===============================
COHERE_API_KEY = ""# ⚠️ Replace with your actual key
PDF_FOLDER = "data"

print("Step 2: Configuration set.")


Step 2: Configuration set.


In [3]:
# ===============================
# 3) LOAD PDFs
# ===============================
def load_pdfs(folder):
    texts = []
    for file in os.listdir(folder):
        if file.endswith(".pdf"):
            with open(os.path.join(folder, file), "rb") as f:
                reader = PyPDF2.PdfReader(f)
                for page in reader.pages:
                    texts.append(page.extract_text() or "")
    print(f"Loaded {len(texts)} pages from PDFs.")
    return texts

texts = load_pdfs(PDF_FOLDER)
print("Step 3: PDFs loaded.")

Loaded 927 pages from PDFs.
Step 3: PDFs loaded.


In [4]:
# ===============================
# 4) SPLIT TEXTS INTO CHUNKS
# ===============================
def split_texts(texts, chunk_size=300):
    chunks = []
    for text in texts:
        words = text.split()
        for i in range(0, len(words), chunk_size):
            chunks.append(" ".join(words[i:i+chunk_size]))
    print(f"Text split into {len(chunks)} chunks.")
    return chunks

chunks = split_texts(texts)
print("Step 4: Texts split into chunks.")

Text split into 1059 chunks.
Step 4: Texts split into chunks.


In [5]:
# ===============================
# 5) TRANSFORM CHUNKS (CLEANING)
# ===============================
def transform_chunks(chunks):
    transformed = []
    for chunk in chunks:
        clean_chunk = " ".join(chunk.split())
        transformed.append(clean_chunk)
    print("Chunks transformed (cleaned).")
    return transformed

chunks = transform_chunks(chunks)
print("Step 5: Chunks transformed.")


Chunks transformed (cleaned).
Step 5: Chunks transformed.


In [6]:
# ===============================
# 6) CREATE VOCAB FOR TYPO CORRECTION
# ===============================
vocab = set(" ".join(chunks).split())
print(f"Vocabulary for typo correction created with {len(vocab)} words.")


# ===============================
# 7) SPELL CORRECTION FUNCTION
# ===============================
def correct_question(question):
    words = question.split()
    corrected_words = []

    for w in words:
        result = process.extractOne(w, vocab, scorer=fuzz.ratio)
        if result is not None:
            match_word, score, *_ = result
            if score >= 80:
                corrected_words.append(match_word)
            else:
                corrected_words.append(w)
        else:
            corrected_words.append(w)
    
    return " ".join(corrected_words)
print("Step 6: Spell correction function defined.")

Vocabulary for typo correction created with 20126 words.
Step 6: Spell correction function defined.


In [7]:


# ===============================
# 8) CREATE VECTOR DATABASE
# ===============================
def create_vector_db(chunks):
    model = SentenceTransformer("paraphrase-MiniLM-L3-v2")
    embeddings = model.encode(chunks, normalize_embeddings=True).astype("float32")
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    print(f"Vector database created with {len(chunks)} embeddings.")
    return index, model, embeddings

index, model, embeddings = create_vector_db(chunks)
print("Step 7: Vector database created.")

Vector database created with 1059 embeddings.
Step 7: Vector database created.


In [8]:
# ===============================
# 9) INITIALIZE COHERE CLIENT
# ===============================
cohere_client = cohere.Client(COHERE_API_KEY)
print("Cohere client initialized.")
print("All steps completed successfully.")

Cohere client initialized.
All steps completed successfully.


In [9]:
# ===============================
# 10) ASK QUESTION FUNCTION
# ===============================
def ask_question(question):
    print("Received question:", question)
    
    # Step 1: Correct typos
    corrected_question = correct_question(question)
    print("Question after typo correction:", corrected_question)
    
    # Step 2: Encode corrected question
    q_vec = model.encode([corrected_question], normalize_embeddings=True).astype("float32")

    # Step 3: Retrieve top 5 chunks
    top_k = 5
    distances, ids = index.search(q_vec, top_k)
    print(f"Retrieved top {top_k} relevant chunks.")

    # Step 4: Transform selected chunks
    selected_chunks = [chunks[i] for i in ids[0]]
    transformed_context = " ".join(selected_chunks)
    print("Context prepared for LLM.")

    # Step 5: Prompt for Cohere
    prompt = f"Answer ONLY using the context below.\n\nContext:\n{transformed_context}\n\nQuestion:\n{question}"
    
    response = cohere_client.chat(
        message=prompt,
        model="command-r-08-2024",
        max_tokens=300,
        temperature=0
    )
    print("Received response from Cohere.")
    return response.text
print("Step 8: Ask question function defined.")

Step 8: Ask question function defined.


In [11]:
# ===============================
# 11) GRADIO UI
# ===============================
app = gr.Interface(
    fn=ask_question,
    inputs=gr.Textbox(lines=4, placeholder="Ask a question about your PDFs..."),
    outputs=gr.Textbox(lines=15),
    title="Multi-Document PDF Chatbot",
)

print("Gradio interface ready. Launching app...")
app.launch()


Gradio interface ready. Launching app...
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
